# R7 — AutoML sobre exactamente los mismos splits

## Correccion

La iteracion anterior comparaba AutoGluon contra XGBoost bajo **protocolos distintos**: AutoML
entrenaba con 95 % de filas sintéticas y evaluaba solo sobre originales,
mientras XGBoost usaba holdouts extraídos de la misma distribución de
entrenamiento. Concluir de ahí que "la experimentación dirigida supera a la
búsqueda automatizada" atribuye al framework lo que puede ser desplazamiento de
distribución.

Aquí AutoGluon recibe **el mismo train, la misma validation y el mismo test**
que el pipeline dirigido, y se mide con las mismas métricas. La comparación
pasa a ser interpretable.

In [1]:
import sys, os

AQUI = os.getcwd()                   
if AQUI not in sys.path:
    sys.path.insert(0, AQUI)

import numpy as np
import pandas as pd
import vishing_common as vc

vc.set_all_seeds()                  
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("dataset:", vc.RAW_FILENAME, "(esquema", vc.DATASET_VERSION + ")")
print("bucket :", vc.BUCKET, "| prefijo:", vc.PREFIX)
print("seed   :", vc.SEED, "| split:", vc.SPLIT_MODE, "| política:", vc.FEATURE_POLICY)
print("xgboost se ejecutará en:", vc.xgb_device())
vc.check_versions()

dataset: biocatch_sinthetic_data_v3.csv (esquema v3)
bucket : poc-vishing | prefijo: v2
seed   : 42 | split: grouped | política: audited
xgboost se ejecutará en: cuda
AVISO: versiones fuera del rango declarado en requirements.txt:
paquete instalada   esperado    ok
xgboost     3.2.0 >=2.0,<2.2 False
  pip install -r requirements.txt


,paquete,instalada,esperado,ok
0,numpy,1.26.4,">=1.26,<2.1",True
1,pandas,2.2.3,">=2.1,<2.3",True
2,scipy,1.12.0,">=1.11,<1.15",True
3,sklearn,1.4.0,">=1.4,<1.6",True
4,imblearn,0.12.4,">=0.12,<0.13",True
5,xgboost,3.2.0,">=2.0,<2.2",False


In [2]:
from autogluon.tabular import TabularPredictor

cfg = vc.read_json(vc.P.feature_contract)
contract = cfg["contratos"][cfg["activo"]]
cols = contract["features"] + [vc.TARGET]

USAR_AUMENTADO = True
tr = vc.read_parquet(vc.P.train_augmented if USAR_AUMENTADO else vc.P.train)[cols]
va = vc.read_parquet(vc.P.val)[cols]
te = vc.read_parquet(vc.P.test)[cols]

print("train:", tr.shape, "| val:", va.shape, "| test:", te.shape)
print("tasa de vishing -> train %.4f | val %.4f | test %.4f"
      % (tr[vc.TARGET].mean(), va[vc.TARGET].mean(), te[vc.TARGET].mean()))

train: (560134, 45) | val: (19673, 45) | test: (20193, 45)
tasa de vishing -> train 0.0188 | val 0.0508 | test 0.0491


In [3]:
import shutil

TIME_LIMIT = 5400   # presupuesto: 90 minutos
RUTA_AG = "/tmp/autogluon_%s" % vc.PREFIX

# AutoGluon reutiliza el directorio si existe y mezcla modelos de corridas
# anteriores en el ensamblado. Se limpia para que la corrida sea la de hoy.
shutil.rmtree(RUTA_AG, ignore_errors=True)

pred = TabularPredictor(
    label=vc.TARGET, problem_type="binary",
    eval_metric="average_precision", path=RUTA_AG,
).fit(
    train_data=tr, tuning_data=va, use_bag_holdout=True,
    presets="best_quality", time_limit=TIME_LIMIT,
    num_bag_folds=8, num_stack_levels=1,
    ag_args_fit={"random_seed": vc.SEED},
)

Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.10.20
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Fri Jul 24 13:35:20 UTC 2026
CPU Count:          4
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda version from torch
Memory Avail:       12.15 GB / 15.42 GB (78.8%)
Disk Space Avail:   7.61 GB / 7.71 GB (98.8%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, nu

> **Límite de reproducibilidad.** AutoGluon con
> `time_limit` decide qué modelos entrena según el tiempo que le sobra, y ese
> tiempo depende de la instancia y de su carga. Dos corridas del mismo notebook
> en instancias distintas pueden producir ensamblados distintos. La semilla fija
> el remuestreo interno, no el presupuesto. Para el resto del pipeline la
> reproducibilidad es exacta; aquí es reproducibilidad **de protocolo**: mismos
> splits, mismas features, mismo presupuesto, mismas métricas.

In [4]:
lb = pred.leaderboard(te, silent=True)
display(lb.head(15))
vc.write_csv(lb, vc.P.automl_leaderboard)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,0.827615,0.837260,average_precision,45.290632,43.650198,3899.208373,0.003472,0.004980,1.348741,3,True,14
1,WeightedEnsemble_L2,0.823522,0.833986,average_precision,44.613668,42.755917,3269.211069,0.002367,0.004963,0.330049,2,True,4
2,LightGBMXT_BAG_L1,0.822133,0.833621,average_precision,29.042721,27.663429,2178.234163,29.042721,27.663429,2178.234163,1,True,1
3,CatBoost_BAG_L2,0.821659,0.833380,average_precision,44.864572,43.090672,3829.364186,0.136647,0.213122,386.107341,2,True,9
4,LightGBM_BAG_L1,0.821495,0.828985,average_precision,15.568580,15.087526,1090.646857,15.568580,15.087526,1090.646857,1,True,2
5,XGBoost_BAG_L2,0.817723,0.826133,average_precision,45.174889,43.458367,3524.457433,0.446964,0.580817,81.200589,2,True,12
6,LightGBM_BAG_L2,0.816387,0.825726,average_precision,44.998469,43.245594,3498.427187,0.270544,0.368044,55.170343,2,True,6
7,RandomForestEntr_BAG_L2,0.815265,0.823644,average_precision,45.022516,43.189202,3727.046043,0.294591,0.311652,283.789198,2,True,8
8,ExtraTreesEntr_BAG_L2,0.814945,0.825961,average_precision,45.277821,43.453261,3550.105594,0.549896,0.575711,106.848750,2,True,11
9,LightGBMXT_BAG_L2,0.814401,0.830170,average_precision,45.150512,43.432096,3511.752290,0.422588,0.554546,68.495446,2,True,5


  escrito s3://poc-vishing/v2/06_results/automl/leaderboard.csv  (14 filas)


's3://poc-vishing/v2/06_results/automl/leaderboard.csv'

In [5]:
# Métricas comparables: umbral fijado sobre validation, medición sobre test.
pv = pred.predict_proba(va)[1].values
pt = pred.predict_proba(te)[1].values

thr = vc.f1_optimal_threshold(va[vc.TARGET].values, pv)
rep_automl = vc.full_report(te[vc.TARGET].values, pt, thr, n_boot=1000, seed=vc.SEED)
vc.print_report(rep_automl, "AutoGluon — TEST (umbral fijado sobre validation)")


  AutoGluon — TEST (umbral fijado sobre validation)
----------------------------------------------------------------
  n = 20,193   positivos = 992 (4.91%)   umbral = 0.3818
----------------------------------------------------------------
  pr_auc    : 0.8276  [0.8073, 0.8488]
  roc_auc   : 0.9622  [0.9547, 0.9693]
  recall    : 0.7490  [0.7218, 0.7762]
  precision : 0.8283  [0.8046, 0.8509]
  f1        : 0.7867  [0.7668, 0.8070]
  matriz    : TN=19,047  FP=154  FN=249  TP=743
  recall@P=0.90: 0.6442
  por 100k sesiones: 4442 alertas, 763 falsas, 1233 vishing perdido
  PR-AUC de referencia (clasificador aleatorio): 0.0491


In [6]:
dirigido = vc.read_json(vc.P.final_test)["test"]

comp = pd.DataFrame({
    "dirigido (XGBoost)": {k: round(dirigido[k], 4) for k in
                           ["pr_auc", "roc_auc", "recall", "precision", "f1"]},
    "AutoGluon": {k: round(rep_automl[k], 4) for k in
                  ["pr_auc", "roc_auc", "recall", "precision", "f1"]},
})
comp.loc["recall_p90"] = [
    round(dirigido["recall_at_precision_90"]["recall"], 4),
    round(rep_automl["recall_at_precision_90"]["recall"], 4),
]
display(comp)

print()
print("Ahora sí es una comparación válida: mismos splits, mismo contrato de")
print("features, mismo criterio de umbral y mismas métricas. Cualquier")
print("diferencia es atribuible al método, no al protocolo de evaluación.")
vc.write_json({"automl_test": rep_automl, "comparacion": comp.to_dict()},
              vc.P.automl_dir + "/comparacion.json")

,dirigido (XGBoost),AutoGluon
pr_auc,0.8458,0.8276
roc_auc,0.9673,0.9622
recall,0.7903,0.7490
precision,0.8025,0.8283
f1,0.7963,0.7867
recall_p90,0.6472,0.6442



Ahora sí es una comparación válida: mismos splits, mismo contrato de
features, mismo criterio de umbral y mismas métricas. Cualquier
diferencia es atribuible al método, no al protocolo de evaluación.
  escrito s3://poc-vishing/v2/06_results/automl/comparacion.json


's3://poc-vishing/v2/06_results/automl/comparacion.json'